In [ ]:
import os, sys
from pathlib import Path
sys.path.append('../src')
sys.path.append('../src/utils/')
from spiking_dataloader import WISDM_spiking_dataloader, WisdmDatasetParser
from output_process import OutputProcess
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

In [2]:
from lava.proc.lif.process import LIF
from lava.proc.dense.process import Dense, LearningDense
from lava.utils.weightutils import SignMode
from lif_mod import LIFEncoder
from dense_mod import DenseEncoder

#path = f"{Path.home()}/snntorch_network/notebook/Trained/network_best.npz"
path = f"{Path.home()}/snntorch_network/nni_experiments/Inibitory_lif_no_encoder/results/hjulef4s/trials/RHEsE/Trained/network_best.npz"
data = np.load(path,allow_pickle=True)


linear1_w= data['linear1']
leaky1_vth= data['leaky1_vth']
leaky1_betas= 1-data['leaky1_betas'] 
leaky1_betas= leaky1_betas if leaky1_betas >= 0 else np.zeros(leaky1_betas.shape)
print(f"leaky1_betas: {leaky1_betas}")
print(f"leaky1_vth: {leaky1_vth}")
linear2_w = data['linear2']
leaky2_vth= data['recurrent_vth']
leaky2_betas= 1 - data['recurrent_betas']
leaky2_betas= leaky2_betas if  leaky2_betas >= 0 else np.zeros(leaky2_betas.shape)
print(f"leaky2_betas: {leaky2_betas}")
print(f"leaky2_vth: {leaky2_vth}")

recurrent_in_weights = data['input_dense']
recurrent_out_weights = - data['output_dense']
recurrent_vth = data['activation_vth']
recurrent_leaky_betas = 1 - data['activation_betas']
recurrent_leaky_betas= recurrent_leaky_betas if recurrent_leaky_betas >= 0 else np.zeros(recurrent_leaky_betas.shape)
print(f"recurrent_leaky_betas: {recurrent_leaky_betas}")
print(f"recurrent_vth: {recurrent_vth}")

linear3_w = data['linear3']
leaky3_vth= data['leaky2_vth']
leaky3_betas= 1 - data['leaky2_betas']
leaky3_betas= leaky3_betas if leaky3_betas >= 0 else np.zeros(leaky3_betas.shape)
print(f"leaky3_betas: {leaky3_betas}")
print(f"leaky3_vth: {leaky3_vth}")


leaky1_betas: 0.8172026127576828
leaky1_vth: 1.711238145828247
leaky2_betas: 0.1649174690246582
leaky2_vth: 0.5812320113182068
recurrent_leaky_betas: 0.13742202520370483
recurrent_vth: 1.1815392971038818
leaky3_betas: 0.0
leaky3_vth: 1.5450438261032104


In [3]:

from lava.magma.core.run_conditions import RunSteps
from lava.magma.core.run_configs import Loihi1SimCfg, Loihi2SimCfg


In [4]:
import json
def deserialize_dict(json_str):
    def convert(obj):
        if isinstance(obj, list):
            return np.array(obj).astype(np.int64)
        if isinstance(obj, int):
            return np.int64(obj)
        return obj
    
    return json.loads(json_str, object_hook=lambda d: {k: convert(v) for k, v in d.items()})


In [5]:
with open('network_fixed.json', 'r') as json_file:
    loaded_json_str = json_file.read()
converted_params = deserialize_dict(loaded_json_str)

In [6]:
total_sample = 4000
signal_step = 40
clear_intervall = 4
train_percentage = 0.6
time_steps = signal_step + clear_intervall

dataset = WisdmDatasetParser('../data/data_watch_subset_0_40.npz', norm=None, class_sublset='custom', subset_list=[0, 4, 6, 8, 9, 10, 14])
val_set = dataset.get_validation_set(shuffle=False, subset=total_sample)
#val_set = dataset.get_validation_set()
train_set = (val_set[0][:int(train_percentage*val_set[0].shape[0])], val_set[1][:int(train_percentage*val_set[0].shape[0])])
val_set = (val_set[0][int(train_percentage*val_set[0].shape[0]):], val_set[1][int(train_percentage*val_set[0].shape[0]):])

num_samples = train_set[0].shape[0]
spiking_loader = WISDM_spiking_dataloader(train_set ,clear_intervall=clear_intervall)
out_sink = OutputProcess(7,num_samples,time_steps, 0)

(6,)
(6,)
ytrain shape (55404, 18)
yval shape (18468, 18)
ytest shape (18469, 18)
num classes train dataset: 7 occurrences of each class:[3127 3044 3102 3047 3150 3087 2973]
num classes eval dataset: 7 occurrences of each class:[1035 1048 1122  996 1110 1053 1007]
num classes test dataset: 7 occurrences of each class:[1046 1048 1046 1036 1076 1026  982]


In [7]:
from lava.proc.learning_rules.stdp_learning_rule import STDPLoihi as STDP

s_stdp = STDP(learning_rate=7,
            A_plus=8,
            A_minus=-4,
            tau_plus=5,
            tau_minus=3,
            t_epoch=40,
            x1_impulse= 1,
            y1_impulse= 1,
            )

In [8]:
linear1 = DenseEncoder(weights=linear1_w, num_message_bits=32, name="linear1")

leaky1 = LIFEncoder(shape=(linear1_w.shape[0],),
                    u = np.zeros(linear1_w.shape[0]),
                    v = np.zeros(linear1_w.shape[0]),
                    du = 1.0,
                    dv = leaky1_betas,
                    vth=leaky1_vth,
                    log_config=0,
                    name= "leaky1"
                )
linear1.a_out.connect(leaky1.a_in)
name = "linear2"
linear2 = Dense(**converted_params[name],
                sign_mode=SignMode.MIXED, name=name)

linear2.s_in.connect_from(leaky1.s_out)

name = "leaky2"
leaky2 = LIF(shape=(linear2_w.shape[0],),
                    **converted_params[name],
                    name= name
                )
#sum.a_out.connect(leaky2.a_in)
linear2.a_out.connect(leaky2.a_in)
#leaky2.a_in.connect_from(linear2.a_out)
name = "recurrent_in"
recurrent_in = Dense(**converted_params[name],
                     name=name)

leaky2.s_out.connect(recurrent_in.s_in)

name = "inibitory_leaky"
ahpc = LIF(shape=(recurrent_in_weights.shape[0],),
                    **converted_params[name],
                    log_config=0,
                    name= name
                )

recurrent_in.a_out.connect(ahpc.a_in)

name = "recurrent_out"
recurrent_out = Dense(**converted_params[name],
                       name=name)
recurrent_out.s_in.connect_from(ahpc.s_out)
recurrent_out.a_out.connect(leaky2.a_in)

name = "linear3"
linear3 = LearningDense(**converted_params[name],
                        learning_rule=s_stdp,
                        name=name)

linear3.s_in.connect_from(leaky2.s_out)
name = "leaky3"
leaky3 = LIF(shape=(linear3_w.shape[0],),
                    **converted_params[name],
                    log_config=0,
                    name= name
                )
leaky3.s_out.connect(linear3.s_in_bap)
leaky3.a_in.connect_from(linear3.a_out)



In [9]:
spiking_loader.data_out.connect(linear1.s_in)
leaky3.s_out.connect(out_sink.spikes_in)
out_sink.label_in.connect_from(spiking_loader.label_out)

In [10]:
clock = tqdm(range(num_samples))
for _, i in enumerate(clock):
      out_sink.run(condition=RunSteps(num_steps=time_steps),
                  run_cfg=Loihi1SimCfg(select_sub_proc_model=True,
                  select_tag='fixed_pt'))
      leaky1.v.set(np.zeros(leaky1.v.shape))
      leaky1.u.set(np.zeros(leaky1.u.shape))
      leaky2.v.set(np.zeros(leaky2.v.shape))
      leaky2.u.set(np.zeros(leaky2.u.shape))
      leaky3.v.set(np.zeros(leaky3.v.shape))
      leaky3.u.set(np.zeros(leaky3.u.shape))
      ahpc.v.set(np.zeros(ahpc.v.shape))
      ahpc.u.set(np.zeros(ahpc.u.shape))
      linear1.a_buff.set(np.zeros(linear1.a_buff.shape))
      linear2.a_buff.set(np.zeros(linear2.a_buff.shape))
      linear3.a_buff.set(np.zeros(linear3.a_buff.shape))

      tmp_predicition = out_sink.pred_labels.get().astype(int)
      current_accuracy = np.sum((tmp_predicition[:i] == train_set[1][:i])/(i+1))*100
      clock.set_description(f"Current accuracy: {current_accuracy:.4f}")
      updated_weights = linear3.weights.get()

# Stop the execution
out_sink.stop()


Current accuracy: 13.9167: 100%|██████████| 2400/2400 [06:00<00:00,  6.66it/s]


In [11]:
print(f"wheights before: {updated_weights}")

wheights before: [[ 59. 255. 255. ... 199. 255. 255.]
 [ 59. 255. 255. ... 199. 255. 255.]
 [ 59. 255. 255. ... 199. 255. 255.]
 ...
 [ 59. 255. 255. ... 199. 255. 255.]
 [ 59. 255. 255. ... 199. 255. 255.]
 [ 59. 255. 255. ... 199. 255. 255.]]


In [12]:
converted_params['linear3']['weights'] = updated_weights

In [13]:
del linear1
del leaky1
del linear2
del leaky2
del recurrent_in
del ahpc
del recurrent_out
del linear3
del leaky3
del out_sink
del spiking_loader

In [14]:
linear1 = DenseEncoder(weights=linear1_w, num_message_bits=32, name="linear1")

leaky1 = LIFEncoder(shape=(linear1_w.shape[0],),
                    u = np.zeros(linear1_w.shape[0]),
                    v = np.zeros(linear1_w.shape[0]),
                    du = 1.0,
                    dv = leaky1_betas,
                    vth=leaky1_vth,
                    log_config=0,
                    name= "leaky1"
                )
linear1.a_out.connect(leaky1.a_in)
name = "linear2"
linear2 = Dense(**converted_params[name],
                sign_mode=SignMode.MIXED, name=name)

linear2.s_in.connect_from(leaky1.s_out)

name = "leaky2"
leaky2 = LIF(shape=(linear2_w.shape[0],),
                    **converted_params[name],
                    name= name
                )
#sum.a_out.connect(leaky2.a_in)
linear2.a_out.connect(leaky2.a_in)
#leaky2.a_in.connect_from(linear2.a_out)
name = "recurrent_in"
recurrent_in = Dense(**converted_params[name],
                    name=name)

leaky2.s_out.connect(recurrent_in.s_in)

name = "inibitory_leaky"
ahpc = LIF(shape=(recurrent_in_weights.shape[0],),
                    **converted_params[name],
                    log_config=0,
                    name= name
                )

recurrent_in.a_out.connect(ahpc.a_in)
name = "recurrent_out"
recurrent_out = Dense(**converted_params[name],
                    name=name)
recurrent_out.s_in.connect_from(ahpc.s_out)
recurrent_out.a_out.connect(leaky2.a_in)

name = "linear3"
linear3 = Dense(**converted_params[name],
                        name=name)

linear3.s_in.connect_from(leaky2.s_out)
name = "leaky3"
leaky3 = LIF(shape=(linear3_w.shape[0],),
                    **converted_params[name],
                    log_config=0,
                    name= name
                )
leaky3.a_in.connect_from(linear3.a_out)




num_samples = val_set[0].shape[0]
spiking_loader = WISDM_spiking_dataloader(val_set ,clear_intervall=clear_intervall)
out_sink = OutputProcess(7,num_samples,time_steps, 0)

spiking_loader.data_out.connect(linear1.s_in)
leaky3.s_out.connect(out_sink.spikes_in)
out_sink.label_in.connect_from(spiking_loader.label_out)

In [15]:
clock = tqdm(range(num_samples))
for _, i in enumerate(clock):
    out_sink.run(condition=RunSteps(num_steps=time_steps),
                run_cfg=Loihi1SimCfg(select_sub_proc_model=True,
                select_tag='fixed_pt'))
    leaky1.v.set(np.zeros(leaky1.v.shape))
    leaky1.u.set(np.zeros(leaky1.u.shape))
    leaky2.v.set(np.zeros(leaky2.v.shape))
    leaky2.u.set(np.zeros(leaky2.u.shape))
    leaky3.v.set(np.zeros(leaky3.v.shape))
    leaky3.u.set(np.zeros(leaky3.u.shape))
    ahpc.v.set(np.zeros(ahpc.v.shape))
    ahpc.u.set(np.zeros(ahpc.u.shape))
    linear1.a_buff.set(np.zeros(linear1.a_buff.shape))
    linear2.a_buff.set(np.zeros(linear2.a_buff.shape))
    linear3.a_buff.set(np.zeros(linear3.a_buff.shape))
    
    tmp_predicition = out_sink.pred_labels.get().astype(int)
    current_accuracy = np.sum((tmp_predicition[:i] == val_set[1][:i])/(i+1))*100
    clock.set_description(f"Current accuracy: {current_accuracy:.4f}")
    updated_weights = linear3.weights.get()

# Stop the execution
out_sink.stop()


100%|██████████| 1600/1600 [03:29<00:00,  7.63it/s]


In [16]:
print(f"current_accuracy: {current_accuracy}")

current_accuracy: 14.1875
